In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn import metrics
from sklearn import ensemble

In [2]:
df = pd.read_csv(r'C:\Users\user\Desktop\Skilfactory\data1\_train_sem09.csv')

In [3]:
df

,Activity,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,D1767,D1768,D1769,D1770,D1771,D1772,D1773,D1774,D1775,D1776
0,1,0.000000,0.497009,0.10,0.0,0.132956,0.678031,0.273166,0.585445,0.743663,...,0,0,0,0,0,0,0,0,0,0
1,1,0.366667,0.606291,0.05,0.0,0.111209,0.803455,0.106105,0.411754,0.836582,...,1,1,1,1,0,1,0,0,1,0
2,1,0.033300,0.480124,0.00,0.0,0.209791,0.610350,0.356453,0.517720,0.679051,...,0,0,0,0,0,0,0,0,0,0
3,1,0.000000,0.538825,0.00,0.5,0.196344,0.724230,0.235606,0.288764,0.805110,...,0,0,0,0,0,0,0,0,0,0
4,0,0.100000,0.517794,0.00,0.0,0.494734,0.781422,0.154361,0.303809,0.812646,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3746,1,0.033300,0.506409,0.10,0.0,0.209887,0.633426,0.297659,0.376124,0.727093,...,0,0,0,0,0,0,0,0,0,0
3747,1,0.133333,0.651023,0.15,0.0,0.151154,0.766505,0.170876,0.404546,0.787935,...,0,0,1,0,1,0,1,0,0,0
3748,0,0.200000,0.520564,0.00,0.0,0.179949,0.768785,0.177341,0.471179,0.872241,...,0,0,0,0,0,0,0,0,0,0
3749,1,0.100000,0.765646,0.00,0.0,0.536954,0.634936,0.342713,0.447162,0.672689,...,0,0,0,0,0,0,0,0,0,0


In [4]:
X, y = df.drop('Activity', axis=1), df['Activity']

In [5]:
y.value_counts(normalize=True)
# Классы сбалансированы

Activity
1    0.542255
0    0.457745
Name: proportion, dtype: float64

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3) 

# 1. Модель логистической регрессии

Основные параметры LogisticRegression:

* random_state — число, на основе которого происходит генерация случайных чисел.
* penalty — метод регуляризации. Возможные значения:
    * 'l1' — L1-регуляризация;
    * 'l2' — L2-регуляризация (используется по умолчанию);
    * 'elasticnet' — эластичная сетка (L1+L2);
    * None — отсутствие регуляризации.
* C — коэффициент обратный коэффициенту регуляризации, то есть равен . Чем больше C, тем меньше регуляризация. По умолчанию C=1, тогда α=1.
* solver — численный метод оптимизации функции потерь logloss, может быть:
    * 'sag' — стохастический градиентный спуск (нужна стандартизация/нормализация);
    * 'saga' — [модификация](https://arxiv.org/pdf/1407.0202.pdf) предыдущего, которая поддерживает работу с негладкими функциями (нужна стандартизация/нормализация);
    * 'newton-cg' — [метод Ньютона с модификацией сопряжённых градиентов](https://docs.scipy.org/doc/scipy/tutorial/optimize.html#newton-conjugate-gradient-algorithm-method-newton-cg) (не нужна стандартизация/нормализация);
    * 'lbfgs' — [метод Бройдена — Флетчера — Гольдфарба — Шанно](https://ru.wikipedia.org/wiki/Алгоритм_Бройдена_—_Флетчера_—_Гольдфарба_—_Шанно) (не нужна стандартизация/нормализация; используется по умолчанию, так как из всех методов теоретически обеспечивает наилучшую сходимость);
    * 'liblinear' — [метод покоординатного спуска](http://www.machinelearning.ru/wiki/index.php?title=Метод_покоординатного_спуска) (не нужна стандартизация/нормализация).
* max_iter — максимальное количество итераций, выделенных на сходимость.

Посмотрим на результаты самого простого исполнения логистической регрессии

In [7]:
# Логистическая регрессия
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

# Делаем предсказания
y_predict_train = log_reg.predict(X_train)
y_predict_test = log_reg.predict(X_test)

# Результаты
print(f'Значение F1-score на тренировочной выборке: {np.round(metrics.f1_score(y_train, y_predict_train), 3)}')
print(f'Значение F1-score на тестовой выборке {np.round(metrics.f1_score(y_test, y_predict_test), 3)}')


Значение F1-score на тренировочной выборке: 0.897
Значение F1-score на тестовой выборке 0.79


Вывод: 
* F1-score на тренировочной выборке 0.897
* F1-score на тестовой выборке 0.79

Модель переобучена, метрики достаточно сильно отличаются на разных выборках.
Оставим гиперпараметры данной модели без изменений и посмотрим на результаты при обучении на 5 фолдах. Будем использовать StratifiedKFold, чтобы сохранить баланс целевой переменной.

In [8]:
# Создаём объект кросс-валидатора 
kf = model_selection.StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Считаем метрики на кросс-валидации
cv_metrics = model_selection.cross_validate(
    estimator=log_reg,
    X = X_train,
    y = y_train,
    cv=kf,
    scoring='f1',
    return_train_score=True
)
display(cv_metrics)
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics['test_score']), 3)}')
print(f'Значение F1-score на тестовой выборке {np.round(metrics.f1_score(y_test, y_predict_test), 3)}')

{'fit_time': array([0.69601464, 0.62836909, 0.68376756, 1.14306498, 0.71095681]),
 'score_time': array([0.01299334, 0.01201677, 0.01299858, 0.01525331, 0.0140388 ]),
 'test_score': array([0.76678445, 0.75909879, 0.78863233, 0.77647059, 0.77142857]),
 'train_score': array([0.9085285 , 0.90371025, 0.90652557, 0.91166078, 0.90379523])}

Среднее значение F1-score на валидационных фолдах: 0.772
Значение F1-score на тестовой выборке 0.79


Валидационная и тестовая выборка показывают примерно одинаковые результаты. Такая модель имеет обощающую способность.

**1.1 GridSearch.**

In [9]:
param_grid = [{'penalty' : ['l1'],
               'solver' : ['liblinear', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]},
              {'penalty': ['l2'],
               'solver' : ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]}]

grid_search_1 = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    cv=kf,
    n_jobs=-1,
    scoring='f1',
    return_train_score=True
    
)
%time grid_search_1.fit(X_train, y_train)


CPU times: total: 2.03 s
Wall time: 4min 34s


,estimator,LogisticRegre...ndom_state=42)
,param_grid,"[{'C': [0.001, 0.01, ...], 'penalty': ['l1'], 'solver': ['liblinear', 'saga']}, {'C': [0.001, 0.01, ...], 'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}]"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,penalty,'l2'


In [10]:
# Получаем полные данные о каждой модели
results_G_S_1 = pd.DataFrame(grid_search_1.cv_results_)
best_index = grid_search_1.best_index_

In [11]:
# Считаем метрики
f1_score_train_G_S_1 = np.round(results_G_S_1.loc[best_index, 'mean_train_score'], 3)
f1_score_test_G_S_1 = np.round(results_G_S_1.loc[best_index, 'mean_test_score'], 3)
print(f'Среднее F1-score на валидационных фолдах лучшей модели: {f1_score_test_G_S_1}')
print(f'Наилучшие значения гиперпараметров: {grid_search_1.best_params_}')
print(f'Значение F1-score на тестовой выборке: {np.round(grid_search_1.score(X_test, y_test), 3)}')

Среднее F1-score на валидационных фолдах лучшей модели: 0.778
Наилучшие значения гиперпараметров: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Значение F1-score на тестовой выборке: 0.794


Значение f1-score на тестовой выборке практически не изменилось.
Отметим, что модель затратила 2 min 7 sec.

**1.2 Использование RandomizedSearchCV для поиска оптимальных гиперпараметров**

In [12]:
from sklearn.model_selection import RandomizedSearchCV

params = [{'penalty' : ['l1'],
               'solver' : ['liblinear', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]},
              {'penalty': ['l2'],
               'solver' : ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
               'C' : [0.001, 0.01, 0.1, 1]}]

random_search_1 = RandomizedSearchCV(
    estimator=log_reg,
    param_distributions=params,
    cv=kf,
    n_iter=15,
    n_jobs=-1,
    return_train_score=True,
    random_state=42
)
%time random_search_1.fit(X_train, y_train)

CPU times: total: 10.9 s
Wall time: 1min 5s


,estimator,LogisticRegre...ndom_state=42)
,param_distributions,"[{'C': [0.001, 0.01, ...], 'penalty': ['l1'], 'solver': ['liblinear', 'saga']}, {'C': [0.001, 0.01, ...], 'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}]"
,n_iter,15
,scoring,None
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [13]:
# Получаем полные данные о каждой модели
results_R_S_1 = pd.DataFrame(random_search_1.cv_results_)
best_index = random_search_1.best_index_

f1_score_train_R_S_1 = np.round(results_R_S_1.loc[best_index, 'mean_train_score'], 3)
f1_score_test_R_S_1 = np.round(results_R_S_1.loc[best_index, 'mean_test_score'], 3)
print(f'Среднее F1-score на валидационных фолдах лучшей модели: {f1_score_test_G_S_1}')
print(f'Лучшие гиперпараметры модели: {random_search_1.best_params_}')
print(f'F1-score на тестовом наборе: {np.round(random_search_1.score(X_test, y_test), 3)}')

Среднее F1-score на валидационных фолдах лучшей модели: 0.778
Лучшие гиперпараметры модели: {'solver': 'saga', 'penalty': 'l2', 'C': 0.1}
F1-score на тестовом наборе: 0.767


Использование RandomizedSearchCV для поиска оптимальных гиперпараметров даёт примерно аналогичные результаты, однако мы потратили меньше времени (45 sec).

# Использование продвинутой оптимизации для поиска гиперпараметров.

**1.3 hyperopt**

In [14]:
import hyperopt
from hyperopt import hp, fmin, tpe, Trials

In [15]:
# зададим пространство поиска гиперпараметров
space = {'C': hp.choice('C', [0.0001, 0.001, 0.01, 0.1, 1]),
         'penalty': hp.choice('penalty', ['l1', 'l2']),
         'solver': hp.choice('solver', ['liblinear', 'saga'])}

In [16]:
random_state=42
# Напишем специальную функцию 
def hyperopt_rf(params, cv=kf, X=X_train, y=y_train, random_state=42):
    params = {'C': float(params['C']),
              'penalty' : str(params['penalty']),
              'solver' : str(params['solver'])}
    # Создаём модель
    model = LogisticRegression(**params, random_state=42, max_iter=5000)
    cv_metrics = model_selection.cross_validate(model, X, y, scoring='f1', cv=kf, return_train_score=True, n_jobs=-1)
    
    test_score_mean = np.mean(cv_metrics['test_score'])
    
    score = test_score_mean
    return -score

In [17]:
%%time
trials = Trials() # Для логирования

best=fmin(hyperopt_rf,
          space=space,
          max_evals=20,
          trials=trials,
          rstate=np.random.default_rng(random_state))
print("Наилучшие значения гиперпараметров {}".format(best))

TPE is being used as the default algorithm.


100%|██████████| 20/20 [03:29<00:00, 10.46s/trial, best loss: -0.7782581740567285]
Наилучшие значения гиперпараметров {'C': 3, 'penalty': 1, 'solver': 0}
CPU times: total: 1.34 s
Wall time: 3min 29s


In [18]:
log_reg_best = LogisticRegression(C=3, penalty='l2', solver='liblinear', max_iter=5000)
cv_metrics_best = model_selection.cross_validate(estimator=log_reg_best,
                                                 X=X_train,
                                                 y=y_train,
                                                 cv=kf,
                                                 scoring='f1',
                                                 return_train_score=True,
                                                 n_jobs=-1)
display(cv_metrics_best)


{'fit_time': array([0.92008448, 1.01621151, 1.04645824, 0.96578336, 1.09596539]),
 'score_time': array([0.03069973, 0.02473927, 0.01550698, 0.02524734, 0.01199841]),
 'test_score': array([0.7654321 , 0.75874126, 0.78507993, 0.77834179, 0.75222816]),
 'train_score': array([0.93327402, 0.92749779, 0.92743363, 0.93492696, 0.93179805])}

In [19]:
log_reg_best.fit(X_train, y_train)
y_test_predict = log_reg_best.predict(X_test)
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics_best['test_score']), 3)}')
print(f'Значение F1-score на тестовой выборке: {np.round(metrics.f1_score(y_test, y_test_predict), 3)}')

Среднее значение F1-score на валидационных фолдах: 0.768
Значение F1-score на тестовой выборке: 0.788


Ситуация примерно аналогична, результаты оптимизации практически не меняеются

**1.4 Оптимизация с помощью optuna**

In [20]:
import optuna
from sklearn.model_selection import cross_val_score

In [21]:
def optuna_rf(trial):
    # Задаём пространство поиска гиперпараметров
    C = trial.suggest_float('C', 0.0001, 1, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'saga'])
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    
    # Модель логистической регрессии
    model = LogisticRegression(random_state=42, max_iter=5000, C=C, solver=solver, penalty=penalty)
    
    scores = cross_val_score(model, X=X_train, y=y_train, cv=kf, scoring='f1', n_jobs=-1)
    score = np.mean(scores)
    
    return score
    
    

In [22]:
study = optuna.create_study(study_name='LogisticRegression', direction='maximize')
study.optimize(optuna_rf, n_trials=30)

[I 2025-11-09 12:31:01,080] A new study created in memory with name: LogisticRegression
[I 2025-11-09 12:31:01,542] Trial 0 finished with value: 0.6966231524436293 and parameters: {'C': 0.001102433412401993, 'solver': 'saga', 'penalty': 'l1'}. Best is trial 0 with value: 0.6966231524436293.
[I 2025-11-09 12:31:02,458] Trial 1 finished with value: 0.7184637792143953 and parameters: {'C': 0.00015915286699231043, 'solver': 'saga', 'penalty': 'l2'}. Best is trial 1 with value: 0.7184637792143953.
[I 2025-11-09 12:31:03,182] Trial 2 finished with value: 0.7776812707448182 and parameters: {'C': 0.0796861357439182, 'solver': 'liblinear', 'penalty': 'l2'}. Best is trial 2 with value: 0.7776812707448182.
[I 2025-11-09 12:31:05,928] Trial 3 finished with value: 0.7470244831682193 and parameters: {'C': 0.0014004036220371771, 'solver': 'saga', 'penalty': 'l2'}. Best is trial 2 with value: 0.7776812707448182.
[I 2025-11-09 12:31:06,472] Trial 4 finished with value: 0.6966231524436293 and parameters

In [23]:
# выводим результаты на обучающей выборке
print(f'Лучшие значения гиперпараметров {study.best_params}')
print(f'f1_score на обучающем наборе: {np.round(study.best_value, 3)}')

Лучшие значения гиперпараметров {'C': 0.04899716104609625, 'solver': 'liblinear', 'penalty': 'l2'}
f1_score на обучающем наборе: 0.781


In [24]:
params_optuna = study.best_params # Передаём параметры
log_reg_best_optuna = LogisticRegression(**params_optuna, random_state=42, max_iter=5000)
# Обучаем модель
log_reg_best_optuna.fit(X_train, y_train)
# Получаем предсказания
y_predict_test_opt = log_reg_best_optuna.predict(X_test)
print(f'Значение F1-score на тестовой выборке: {np.round(metrics.f1_score(y_test, y_predict_test_opt), 3)}')

Значение F1-score на тестовой выборке: 0.794


Результаты использования 4 различных методов оптимизации показывают, что конечный результат на тестовой выборке, которую модель ещё не видела, практически не отличается, однако каждый метод использует различное количество времени на поиск оптимальных параметров. В нашем примере наилучшие значения показывает метод оптимизации с помощью optuna. Простой синтаксис, быстрота оптимизации, а также временные затраты делают этот метод самым эффективным.

**Сводные результаты по всем моделям:**
* GridSearch. Значение F1-score на тестовой выборке: 0.794 
* RandomSearch. Значение F1-score на тестовой выборке: 0.767
* Hyperopt. Значение F1-score на тестовой выборке: 0.788
* Optuna. Значение F1-score на тестовой выборке: 0.791


# 2. Попробуем решить данную проблему с помощью более мощного алгоритма - случайный лес

**Основные параметры RandomForestClassifier:**

* `n_estimators` — количество деревьев в лесу (число K из бэггинга; по умолчанию равно 100);
* `criterion` — критерий информативности разбиения для каждого из деревьев (`'gini'` — критерий Джини и `'entropy'` — энтропия Шеннона; по умолчанию — `'gini'`);
* `max_depth` — максимальная глубина одного дерева (по умолчанию — `None`, то есть глубина дерева не ограничена);
* `max_features` — максимальное число признаков, которые будут использоваться каждым из деревьев (число L из метода случайных подпространств; по умолчанию — `'sqrt'`; для обучения каждого из деревьев используется $\sqrt{m}$ признаков, где $m$ — число признаков в начальном наборе данных);
* `min_samples_leaf` — минимальное число объектов в листе (по умолчанию — 1);
* `random_state` — параметр, отвечающий за генерацию случайных чисел.

Создадим базовую модель, чтобы посмотреть как она решает поставленную задачу.

In [25]:
#Создаём объект класса RandomForestClassifier
random_forest = ensemble.RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    criterion='gini',
    min_samples_leaf=10,
    random_state=42,
    max_features=400,
    n_jobs=-1
)

# Используем кросс-валидацию
cv_metrics_random_forest = model_selection.cross_validate(
    estimator=random_forest,
    X=X_train,
    y=y_train,
    scoring='f1',
    return_train_score=True,
    cv=kf,
    n_jobs=-1
)

display(cv_metrics_random_forest)

random_forest.fit(X_train, y_train)
y_pred_test_forest = random_forest.predict(X_test)
print(f'Среднее значение F1-score на валидационных фолдах: {np.round(np.mean(cv_metrics_random_forest['test_score']), 3)}')
print(f'Значение F1-score на тестовой выборке {np.round(metrics.f1_score(y_test, y_pred_test_forest), 3)}')

{'fit_time': array([4.45520878, 4.50487423, 3.49601936, 4.67439032, 3.78475714]),
 'score_time': array([0.25765443, 0.14097571, 1.19384646, 0.06958175, 0.90510869]),
 'test_score': array([0.78057554, 0.79649123, 0.7915937 , 0.80550775, 0.79292035]),
 'train_score': array([0.85172109, 0.84995587, 0.84632321, 0.85789474, 0.85165794])}

Среднее значение F1-score на валидационных фолдах: 0.793
Значение F1-score на тестовой выборке 0.804


Модель хорошо справляется с классификацией, так как даже на случайных параметрах модель имеет метрику f1-score лучше чем на самой эффективной модели логистической регрессии.

**2.1 GridSearch**
Из-за обилия параметров и сложности алгоритма перебор сеткой займёт очень много времени, поэтому пропустим его и попробуем проверить 40 случайных моделей с помощью RandomSearch

**2.2 RandomSearch**

In [26]:
param_grid_forest = {'n_estimators' : [100, 200, 300, 400, 500],
                     'criterion' : ['gini', 'entropy'],
                     'max_depth' : [4, 5, 7, 8],
                     'max_features' : ['sqrt', 'log2'],
                     'min_samples_leaf' : [6, 7, 8, 9]
                     }

random_search_forest = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_grid_forest,
    cv=kf,
    scoring='f1',
    return_train_score=True,
    n_iter=40,
    n_jobs=-1
)
%time random_search_forest.fit(X_train, y_train)

CPU times: total: 2.03 s
Wall time: 1min 11s


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'criterion': ['gini', 'entropy'], 'max_depth': [4, 5, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [6, 7, ...], ...}"
,n_iter,40
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [27]:
results_R_S_forest = pd.DataFrame(random_search_forest.cv_results_)
best_index = random_search_forest.best_index_

f1_score_train_R_S_forest = results_R_S_forest.loc[best_index, 'mean_train_score']
f1_score_test_R_S_forest = results_R_S_forest.loc[best_index, 'mean_test_score']
print(f'Среднее F1-score на валидационных фолдах лучшей модели: {np.round(f1_score_test_R_S_forest, 3)}')
print(f'Лучшие гиперпараметры модели: {random_search_forest.best_params_}')
print(f'F1-score на тестовом наборе: {np.round(random_search_forest.score(X_test, y_test), 3)}')

Среднее F1-score на валидационных фолдах лучшей модели: 0.784
Лучшие гиперпараметры модели: {'n_estimators': 100, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'max_depth': 8, 'criterion': 'gini'}
F1-score на тестовом наборе: 0.804


Случайный поиск не улучшил целевую метрику, поэтому попробуем найти оптимальные гиперпараметры с помощью продвинутых методов оптимизации.

**2.3 Hyperopt**

In [28]:
space = {'n_estimators': hp.randint('n_estimators', 100, 501),
         'criterion': hp.choice('criterion', ['gini', 'entropy']),
         'max_depth': hp.randint('max_depth', 5, 11),
         'max_features': hp.randint('max_features', 5, 51),
         'min_samples_leaf': hp.randint('min_samples_leaf', 5, 16)}

def hyperopt_rf(params, cv=kf, X=X_train, y=y_train, random_state=42):
    params_dict = {
        'n_estimators': params['n_estimators'],
        'criterion': params['criterion'],  
        'max_depth': params['max_depth'],
        'max_features': params['max_features'],
        'min_samples_leaf': params['min_samples_leaf']
    }
    
    model = ensemble.RandomForestClassifier(**params_dict, random_state=42)
    cv_metrics = model_selection.cross_validate(model, X, y, scoring='f1', cv=kf, return_train_score=True, n_jobs=-1)
    
    score = np.mean(cv_metrics['test_score'])
    return -score


In [29]:
%%time
trials = Trials() # Для логирования

best=fmin(hyperopt_rf,
          space=space,
          max_evals=40,
          trials=trials,
          rstate=np.random.default_rng(random_state))

if best['criterion'] == str(0):
    best['criterion'] = 'gini'
else:
    best['criterion'] = 'entropy'

print(f'Наилучшие значения гиперпараметров {best}')

TPE is being used as the default algorithm.


100%|██████████| 40/40 [02:27<00:00,  3.69s/trial, best loss: -0.7955690340273351]
Наилучшие значения гиперпараметров {'criterion': 'entropy', 'max_depth': 9, 'max_features': 42, 'min_samples_leaf': 5, 'n_estimators': 106}
CPU times: total: 3.44 s
Wall time: 2min 27s


In [30]:
forest_best_hyper = ensemble.RandomForestClassifier(**best, random_state=42)
forest_best_hyper.fit(X_train, y_train)
y_pred_test_forest_best_hyper = forest_best_hyper.predict(X_test)
print(f'F1-score на тестовом наборе: {np.round(metrics.f1_score(y_test, y_pred_test_forest_best_hyper), 3)}')

F1-score на тестовом наборе: 0.809


F1-score увеличился на 0.09 после 40 итераций. Попробуем ещё улучшить F1-score с помощью optuna 

**2.4 Optuna**

In [31]:
def optuna_rf(trial):
    # Задаём пространство поиска гиперпараметров
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)     
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])  
    max_depth = trial.suggest_int('max_depth', 5, 25)              
    max_features = trial.suggest_int('max_features', 5, 200)        
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 25) 
    
    # Модель Случайного леса
    model = ensemble.RandomForestClassifier(random_state=42, n_estimators=n_estimators, criterion=criterion, max_depth=max_depth,
                                            max_features=max_features, min_samples_leaf=min_samples_leaf)
    
    scores = cross_val_score(model, X=X_train, y=y_train, cv=kf, scoring='f1', n_jobs=-1)
    score = np.mean(scores)
    
    return score
    

In [32]:
study = optuna.create_study(study_name='RandomForest', direction='maximize')
study.optimize(optuna_rf, n_trials=40)

[I 2025-11-09 12:35:55,221] A new study created in memory with name: RandomForest
[I 2025-11-09 12:36:04,546] Trial 0 finished with value: 0.785965387647457 and parameters: {'n_estimators': 769, 'criterion': 'entropy', 'max_depth': 25, 'max_features': 46, 'min_samples_leaf': 15}. Best is trial 0 with value: 0.785965387647457.
[I 2025-11-09 12:36:07,789] Trial 1 finished with value: 0.7788321531881275 and parameters: {'n_estimators': 466, 'criterion': 'entropy', 'max_depth': 15, 'max_features': 14, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.785965387647457.
[I 2025-11-09 12:36:17,117] Trial 2 finished with value: 0.7723928643140578 and parameters: {'n_estimators': 718, 'criterion': 'entropy', 'max_depth': 7, 'max_features': 75, 'min_samples_leaf': 23}. Best is trial 0 with value: 0.785965387647457.
[I 2025-11-09 12:36:31,920] Trial 3 finished with value: 0.7859985634086188 and parameters: {'n_estimators': 967, 'criterion': 'entropy', 'max_depth': 8, 'max_features': 74, 'min_s

In [33]:
params_optuna = study.best_params # Передаём параметры
print(params_optuna)
random_forest_best_optuna = ensemble.RandomForestClassifier(**params_optuna, random_state=42)
# Обучаем модель
random_forest_best_optuna.fit(X_train, y_train)
# Получаем предсказания
y_predict_test_opt = random_forest_best_optuna.predict(X_test)
print(f'Значение F1-score на тестовой выборке: {np.round(metrics.f1_score(y_test, y_predict_test_opt), 3)}')

{'n_estimators': 835, 'criterion': 'gini', 'max_depth': 23, 'max_features': 155, 'min_samples_leaf': 5}
Значение F1-score на тестовой выборке: 0.821


Расширение количества итераций и границ промеряемых гиперпараметров позволило значительно повысит целевую метрику f1-score до 0.821

**Сводные результаты по всем моделям:**
* Random Search. F1-score на тестовом наборе: 0.8
* Hyperopt. F1-score на тестовом наборе: 0.809
* Optuna. F1-score на тестовой выборке: 0.821

Мы построили 2 модели - логистическую регрессию и случайный лес, каждую из которых оптимизировали различными методами. Методы отличались по эффективности поиска и времени, которое алгоритмы использовали. Каждый метод имеет свои сильные и слабые стороны, однако, на мой взгляд, можно выделить библиотеку optuna, которая при достаточно простом синтаксисе, имеет достаточно высокую скорость, а что ещё лучше - умный подход поиска оптимальных параметров.